# 06 — Basic RAG (Retrieval Augmented Generation)

RAG lets an AI model answer questions about documents it has never seen before.
Instead of asking the model to guess, we find the most relevant passages from the document first and hand them to the model as context.

In this notebook we will:
1. Download a PDF report
2. Split it into small chunks of text
3. Convert each chunk into a **vector embedding** (a list of numbers that captures meaning)
4. Store the embeddings in a **vector database** (Qdrant)
5. Ask a question — find the closest chunks and let the LLM answer from them

Run each cell from top to bottom by pressing **Shift + Enter**.

In [ ]:
import os
os.environ["UV_EXTRA_INDEX_URL"] = "https://pypi.org/simple"

# enable colour output for all print() calls in this notebook
!uv pip install rich -q
from rich import print, print_json

In [ ]:
!uv pip install qdrant-client pypdf openai requests -q

## Step 1 — Get your API token

We need the MaaS token for both the embedding model and the LLM.

> **Note:** You must have logged in to OpenShift first. If you have not done so, run **Step 1 — Log in to OpenShift** in notebook `01-getting-connected.ipynb` before continuing.

In [ ]:
import base64
import subprocess

result = subprocess.run(
    ["oc", "get", "secret", "maas-secret", "-o", "jsonpath={.data.token}"],
    capture_output=True, text=True
)
TOKEN = base64.b64decode(result.stdout.strip()).decode()
print("Token obtained:", TOKEN[:20], "...")

## Step 2 — Download and parse the PDF

We will use the Woolworths Group 2025 Sustainability Report as our document.
Only the first 30 pages are loaded to keep the demo fast.

In [ ]:
import requests
import pypdf

PDF_URL  = "https://www.woolworthsgroup.com.au/content/dam/wwg/sustainability/reports/2025_WG_Sustainability_Report_Interactive_SPREADS.pdf"
MAX_PAGES = 30

print("Downloading PDF...")
response = requests.get(PDF_URL)
with open("report.pdf", "wb") as f:
    f.write(response.content)
print(f"Saved report.pdf ({len(response.content) // 1024} KB)")

reader = pypdf.PdfReader("report.pdf")
pages  = reader.pages[:MAX_PAGES]

full_text = ""
for page in pages:
    full_text += page.extract_text() or ""

print(f"Extracted text from {len(pages)} pages ({len(full_text):,} characters)")

## Step 3 — Split into chunks

Embedding models work best on short pieces of text.
We slide a window across the full text, creating overlapping 500-character chunks.

In [ ]:
CHUNK_SIZE    = 500
CHUNK_OVERLAP = 50

chunks = []
start  = 0
while start < len(full_text):
    chunks.append(full_text[start : start + CHUNK_SIZE])
    start += CHUNK_SIZE - CHUNK_OVERLAP

print(f"Created {len(chunks)} chunks")
print()
print("First chunk preview:")
print(chunks[0][:300])

## Step 4 — Embed chunks and store in Qdrant

We use the `bge-m3` embedding model from MaaS to convert each chunk into a 1024-dimensional vector.
Qdrant stores the vectors in memory so we can search them by similarity.

This step takes a minute or two — you will see progress printed for each batch.

In [ ]:
from openai import OpenAI
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

EMBED_URL  = "https://maas.apps.ocp.cloud.rhai-tmm.dev/prelude-maas/bge-m3/v1"
EMBED_DIM  = 1024
BATCH_SIZE = 10

embedder = OpenAI(base_url=EMBED_URL, api_key=TOKEN)

# set up Qdrant in memory — no server needed
qdrant = QdrantClient(":memory:")
qdrant.create_collection(
    collection_name="docs",
    vectors_config=VectorParams(size=EMBED_DIM, distance=Distance.COSINE)
)

point_id = 0
for i in range(0, len(chunks), BATCH_SIZE):
    batch = chunks[i : i + BATCH_SIZE]
    response = embedder.embeddings.create(model="bge-m3", input=batch)
    vectors  = [d.embedding for d in response.data]

    qdrant.upsert(
        collection_name="docs",
        points=[
            PointStruct(id=point_id + j, vector=vec, payload={"text": chunk})
            for j, (vec, chunk) in enumerate(zip(vectors, batch))
        ]
    )
    point_id += len(batch)
    print(f"Indexed {point_id}/{len(chunks)} chunks")

print()
print(f"Done — {point_id} chunks stored in Qdrant")

## Step 5 — Ask a question

We embed the question, find the 5 most similar chunks in Qdrant, then pass them to the LLM as context.
The model answers using only the retrieved text — no guessing.

In [ ]:
LLM_URL  = "https://maas.apps.ocp.cloud.rhai-tmm.dev/prelude-maas/qwen36-27b/v1"
QUESTION = "What are Woolworths' key sustainability targets for 2025?"

# embed the question
q_vector = embedder.embeddings.create(model="bge-m3", input=[QUESTION]).data[0].embedding

# find the 5 closest chunks
results = qdrant.query_points(collection_name="docs", query=q_vector, limit=5).points
context = "\n\n".join(r.payload["text"] for r in results)

print(f"Retrieved {len(results)} chunks — sending to LLM...")
print()

# ask the LLM
llm = OpenAI(base_url=LLM_URL, api_key=TOKEN)
answer = llm.chat.completions.create(
    model="qwen36-27b",
    messages=[
        {"role": "system", "content": "You are a helpful assistant. Answer using only the context provided. If the answer is not in the context, say so."},
        {"role": "user",   "content": f"Context:\n{context}\n\nQuestion: {QUESTION}"}
    ]
)

print(answer.choices[0].message.content)